# California Collision Severity Mining — All-Feature Pipeline & Binary Classification

This notebook streams collision records from `mart.vw_raw_collision`, dynamically uses **all available columns** (excluding `TARGET = 'severe_or_fatal'`) with modular helper functions, selects the Top 30 features via a Base Decision Tree, and compares three binary classification models (**Logistic Regression**, **Random Forest**, and **Gaussian Naive Bayes**).

---
## 1. Stream & Clean 500k Records from PostgreSQL (All Features)
Defines `load_clean_data(...)` to stream all columns from `mart.vw_raw_collision` in 100k chunks, drop missing values, dynamically separate `NUM_COLS` and `CAT_COLS` from `FEATURES`, and optimize column dtypes.

In [1]:
import os, gc, time
import numpy as np
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv

load_dotenv()

TARGET = 'severe_or_fatal'

engine = create_engine(
    f"postgresql+psycopg://{os.getenv('PGUSER')}:{os.getenv('PGPASSWORD')}"
    f"@{os.getenv('PGHOST')}:{os.getenv('PGPORT') or 5432}/{os.getenv('PGDATABASE')}",
    echo=False,
)

def load_clean_data(engine, table_name="mart.vw_raw_collision", target_rows=500_000, chunk_size=100_000):
    """Stream all columns from PostgreSQL in chunks, drop NAs, and derive feature lists."""
    t0 = time.perf_counter()
    query = f"SELECT * FROM {table_name}"

    temp, total = [], 0
    with engine.connect().execution_options(stream_results=True) as conn:
        for batch_idx, chunk in enumerate(pd.read_sql(query, conn, chunksize=chunk_size), 1):
            clean = chunk.dropna()
            if not clean.empty:
                temp.append(clean)
                total += len(clean)
            print(f"Batch {batch_idx:02d}: +{len(clean):,} valid rows (Total: {min(total, target_rows):,} / {target_rows:,})")
            if total >= target_rows:
                break

    if not temp:
        raise ValueError(f"No valid rows after dropna() from {table_name}")

    df_clean = pd.concat(temp, ignore_index=True).iloc[:target_rows].copy()
    del temp
    gc.collect()

    features = [c for c in df_clean.columns if c != TARGET]
    num_cols = df_clean[features].select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = df_clean[features].select_dtypes(exclude=[np.number]).columns.tolist()

    if num_cols:
        df_clean[num_cols] = df_clean[num_cols].astype(np.float32)
    if cat_cols:
        df_clean[cat_cols] = df_clean[cat_cols].astype(str)
    df_clean[TARGET] = df_clean[TARGET].astype(np.int8)

    ram_mb = df_clean.memory_usage(deep=True).sum() / 1e6
    print(
        f"\nDone in {time.perf_counter() - t0:.1f}s | Shape: {df_clean.shape} | "
        f"Features: {len(features)} ({len(num_cols)} num, {len(cat_cols)} cat) | "
        f"RAM: {ram_mb:.1f} MB | Severe rate: {df_clean[TARGET].mean() * 100:.2f}%"
    )
    return df_clean, features, num_cols, cat_cols

df_clean, FEATURES, NUM_COLS, CAT_COLS = load_clean_data(engine)


ImportError: DLL load failed while importing _util_cy: An Application Control policy has blocked this file.

**Data Preparation Notes:**
- `load_clean_data` streams all columns from `mart.vw_raw_collision` rather than limiting to a subset.
- Numeric features are detected automatically via `np.number` and downcast to `float32`; categorical, boolean, and date/text columns are cast to strings for sparse one-hot encoding.

---
## 2. Base Model (All Features) & Top 30 Feature Importance
Splits the data before preprocessing to avoid leakage, encodes all features once in a sparse matrix, evaluates the $H_0$ majority baseline vs. a Base Decision Tree using `evaluate_model(...)`, and extracts the Top 30 features via `get_top_k_features(...)`.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, recall_score, f1_score, roc_auc_score, roc_curve

# 1. Stratified 80/20 Train-Test Split BEFORE fitting any preprocessors
X = df_clean[FEATURES]
y = df_clean[TARGET].to_numpy()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# 2. Single Sparse Preprocessing Pass (fitted strictly on training split)
transformers = []
if NUM_COLS:
    transformers.append((StandardScaler(), NUM_COLS))
if CAT_COLS:
    transformers.append((OneHotEncoder(handle_unknown='ignore', sparse_output=True, dtype=np.float32), CAT_COLS))
preprocessor = make_column_transformer(*transformers)

X_train_enc = preprocessor.fit_transform(X_train)
X_test_enc = preprocessor.transform(X_test)
feature_names = np.array([n.split('__')[-1] for n in preprocessor.get_feature_names_out()])
print(f"Encoded matrix: {X_train_enc.shape[0]:,} train rows x {X_train_enc.shape[1]:,} encoded features")

def evaluate_model(name, model, X_tr, y_tr, X_te, y_te, n_features):
    """Fit a binary classifier, compute test metrics, and return metrics, predictions, and probabilities."""
    t0 = time.perf_counter()
    model.fit(X_tr, y_tr)
    elapsed = time.perf_counter() - t0

    y_pred = model.predict(X_te)
    y_prob = model.predict_proba(X_te)[:, 1]
    auc = roc_auc_score(y_te, y_prob)

    metrics = {
        "Model": name,
        "Features": int(n_features),
        "Accuracy": round(accuracy_score(y_te, y_pred), 4),
        "Balanced Acc": round(balanced_accuracy_score(y_te, y_pred), 4),
        "Recall": round(recall_score(y_te, y_pred, zero_division=0), 4),
        "F1": round(f1_score(y_te, y_pred, zero_division=0), 4),
        "ROC-AUC": round(auc, 4),
        "Train Sec": round(elapsed, 2),
    }
    print(
        f"{name:<35} -> ROC-AUC: {metrics['ROC-AUC']:.4f} | "
        f"Balanced Acc: {metrics['Balanced Acc']:.4f} | "
        f"Recall: {metrics['Recall']:.4f} | "
        f"F1: {metrics['F1']:.4f} ({metrics['Train Sec']:.2f}s)"
    )
    return metrics, y_pred, y_prob

def get_top_k_features(model, feature_names, k=30):
    """Extract and plot the top-k most important features from a fitted tree model."""
    top_k = min(k, len(feature_names))
    top_idx = np.argsort(model.feature_importances_)[::-1][:top_k]
    top_imp = pd.Series(model.feature_importances_[top_idx], index=feature_names[top_idx])

    print(f"\n=== TOP {top_k} FEATURES (Base Model) ===")
    print(top_imp.round(5))

    plt.figure(figsize=(10, max(4, int(top_k * 0.27))))
    top_imp.plot(kind='barh', color='steelblue', edgecolor='black').invert_yaxis()
    plt.title(f"Top {top_k} Feature Importances (Base Decision Tree — All Features)")
    plt.xlabel("Gini Importance")
    plt.grid(axis='x', linestyle='--', alpha=0.4)
    plt.tight_layout()
    plt.show()
    return top_idx, top_imp

# 3. H0 Null Baseline vs Base Model (All Features)
h0_metrics, y_pred_h0, y_prob_h0 = evaluate_model(
    "H0 (Null Majority)", DummyClassifier(strategy='most_frequent'),
    X_train_enc, y_train, X_test_enc, y_test, n_features=0
)
base_model = DecisionTreeClassifier(max_depth=8, class_weight='balanced', random_state=42)
base_metrics, y_pred_base, y_prob_base = evaluate_model(
    "Base Decision Tree (All Features)", base_model,
    X_train_enc, y_train, X_test_enc, y_test, n_features=X_train_enc.shape[1]
)

# 4. Extract & Plot Top 30 Features
top30_idx, top30_imp = get_top_k_features(base_model, feature_names, k=30)


**Baseline & Feature Selection Notes:**
- `evaluate_model` standardizes training time measurement and evaluation metrics (`ROC-AUC`, `Balanced Acc`, `Recall`, `F1`) across models.
- `get_top_k_features` ranks every encoded numeric and one-hot categorical column by Gini importance so the downstream binary classifiers run on a compact, high-signal feature subset.

---
## 3. Train & Compare 3 Binary Classification Models on Top 30 Features
Uses `slice_top_features(...)` to extract the Top 30 features into compact `float32` matrices and compares **Logistic Regression**, **Random Forest**, and **Gaussian Naive Bayes** (`GaussianNB`) against the Base Decision Tree and $H_0$ baseline.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB

def slice_top_features(X_tr_enc, X_te_enc, top_idx):
    """Slice sparse encoded matrices to top features and convert to dense float32 arrays."""
    to_dense = lambda m: (m.toarray() if hasattr(m, 'toarray') else np.asarray(m)).astype(np.float32)
    return to_dense(X_tr_enc[:, top_idx]), to_dense(X_te_enc[:, top_idx])

def plot_model_comparison(results_df, roc_curves, y_test):
    """Plot ROC curves and side-by-side metric comparison bars across all evaluated models."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot([0, 1], [0, 1], 'k--', label='H0 Random / Majority (AUC = 0.5000)')
    for name, (probs, auc) in roc_curves.items():
        fpr, tpr, _ = roc_curve(y_test, probs)
        axes[0].plot(fpr, tpr, linewidth=2, label=f"{name} (AUC = {auc:.4f})")
    axes[0].set_title("ROC Curves: All Features vs Top 30 Features")
    axes[0].set_xlabel("False Positive Rate")
    axes[0].set_ylabel("True Positive Rate (Recall)")
    axes[0].legend(loc='lower right', fontsize=9)
    axes[0].grid(alpha=0.3)

    plot_df = results_df.set_index('Model')[['ROC-AUC', 'Balanced Acc', 'Recall']]
    plot_df.plot(kind='barh', ax=axes[1], edgecolor='black')
    axes[1].invert_yaxis()
    axes[1].set_xlim(0, 1.0)
    axes[1].set_title("Evaluation Metrics by Model")
    axes[1].set_xlabel("Score")
    axes[1].grid(axis='x', linestyle='--', alpha=0.4)

    plt.tight_layout()
    plt.show()

# 1. Slice Top 30 Features into compact float32 dense matrices
X_train_top30, X_test_top30 = slice_top_features(X_train_enc, X_test_enc, top30_idx)
top_k = X_train_top30.shape[1]

# 2. Define 3 Binary Classification Models (using Gaussian Naive Bayes as the third model)
top30_models = {
    "Logistic Regression (Top 30)": LogisticRegression(
        class_weight='balanced', max_iter=500, random_state=42
    ),
    "Random Forest (Top 30)": RandomForestClassifier(
        n_estimators=100, max_depth=12, min_samples_leaf=20,
        class_weight='balanced', n_jobs=-1, random_state=42
    ),
    "Gaussian Naive Bayes (Top 30)": GaussianNB(),
}

# 3. Train, Evaluate & Compare
results = [h0_metrics, base_metrics]
roc_curves = {"Base Decision Tree (All Features)": (y_prob_base, base_metrics["ROC-AUC"])}

for name, clf in top30_models.items():
    metrics, _, probs = evaluate_model(
        name, clf, X_train_top30, y_train, X_test_top30, y_test, n_features=top_k
    )
    results.append(metrics)
    roc_curves[name] = (probs, metrics["ROC-AUC"])

results_df = pd.DataFrame(results).sort_values(by='ROC-AUC', ascending=False).reset_index(drop=True)
print("\n=== MODEL COMPARISON (ALL FEATURES VS TOP 30 FEATURES) ===")
display(results_df)

plot_model_comparison(results_df, roc_curves, y_test)


---
## Summary & Next Steps

### Q&A
- **How are all features incorporated?** `load_clean_data` selects all columns from `mart.vw_raw_collision` (`SELECT *`), keeps everything except `TARGET = 'severe_or_fatal'` in `FEATURES`, and classifies numeric vs. categorical columns dynamically.
- **How is the code structured?** Core workflow stages are wrapped into five concise, reusable functions: `load_clean_data`, `evaluate_model`, `get_top_k_features`, `slice_top_features`, and `plot_model_comparison`.
- **Which binary classifiers are compared on the Top 30 features?** Logistic Regression, Random Forest, and Gaussian Naive Bayes (`GaussianNB`) alongside the $H_0$ majority baseline and the all-feature Base Decision Tree.

### Data Analysis Key Findings
- Execute all cells against `mart.vw_raw_collision` to inspect the dynamically detected numeric/categorical feature counts, Top 30 Gini importances, and model comparison metrics (`ROC-AUC`, `Balanced Acc`, `Recall`, `F1`).

### Insights or Next Steps
- Run Cells 1–3 sequentially in your `dwh` kernel and inspect `results_df` to verify whether linear (`LogisticRegression`), tree-ensemble (`RandomForestClassifier`), or probabilistic (`GaussianNB`) decision boundaries achieve the highest recall and `ROC-AUC` on `severe_or_fatal`.
- If high-cardinality ID/timestamp text columns dominate `CAT_COLS` when using `SELECT *`, consider reviewing the Top 30 importance plot to check for identifier overfitting.